# WASB-SBDT — Test de detección de pelota en pádel

Este notebook prueba los modelos pre-entrenados de WASB-SBDT sobre tu vídeo de pádel.
Compara TrackNetV2 y WASB (ambos con pesos de tenis y badminton) y mide el % de frames con pelota detectada.

**Requisitos:** GPU T4 o superior (Colab gratuito es suficiente)

In [ ]:
# Verificar GPU disponible
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NO GPU DETECTADA')

In [ ]:
# Instalar dependencias
!pip install gdown opencv-python-headless tqdm --quiet

In [ ]:
# Clonar WASB-SBDT
import os
if not os.path.exists('WASB-SBDT'):
    !git clone https://github.com/nttcom/WASB-SBDT.git --quiet
    print('Clonado.')
else:
    print('Ya existe.')

os.makedirs('pretrained_weights', exist_ok=True)

In [ ]:
# Descargar pesos pre-entrenados
# Descargamos TrackNetV2 y WASB, tanto tenis como badminton
import gdown

weights = {
    'tracknetv2_tennis':    '1BCnmvDX-LZpbkk4vlMEXMm-uzCoqJzDx',
    'tracknetv2_badminton': '1lCVYzua7jJfuKqvWGypkYqr6PHA1EaPq',
    'wasb_tennis':          '14AeyIOCQ2UaQmbZLNQJa1H_eSwxUXk7z',
    'wasb_badminton':       '17Ac0pO5oryh1JwgwTFQTjOKHY3umbDQu',
}

for name, file_id in weights.items():
    path = f'pretrained_weights/{name}_best.pth.tar'
    if not os.path.exists(path):
        print(f'Descargando {name}...')
        gdown.download(id=file_id, output=path, quiet=False)
    else:
        print(f'{name}: ya descargado.')

In [ ]:
# Subir vídeo de prueba
from google.colab import files
print('Sube tu vídeo de prueba (test_60s.mp4 o similar):')
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f'Vídeo subido: {VIDEO_PATH}')

In [ ]:
# Definir el modelo TrackNetV2 standalone
# (sin depender del sistema de configs de Hydra de WASB)
import torch
import torch.nn as nn
import sys

sys.path.insert(0, 'WASB-SBDT/src')

# UNet parts (copiado de WASB-SBDT para evitar dependencias de Hydra)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, bn_first=True):
        super().__init__()
        layers = []
        if bn_first:
            layers += [nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True), 
                       nn.Conv2d(in_ch, out_ch, 3, padding=1)]
        else:
            layers += [nn.Conv2d(in_ch, out_ch, 3, padding=1),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        layers += [nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                   nn.Conv2d(out_ch, out_ch, 3, padding=1),
                   nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.conv = nn.Sequential(*layers)
    def forward(self, x): return self.conv(x)

class Down(nn.Module):
    def __init__(self, n, in_ch, out_ch, bn_first=True):
        super().__init__()
        layers = [nn.MaxPool2d(2)]
        layers += [DoubleConv(in_ch, out_ch, bn_first=bn_first)]
        for _ in range(n - 1):
            layers += [DoubleConv(out_ch, out_ch)]
        self.mp = nn.Sequential(*layers)
    def forward(self, x): return self.mp(x)

class Up(nn.Module):
    def __init__(self, n, in_ch, skip_ch, out_ch, bilinear=True, mode='nearest', bn_first=True, halve_channel=False):
        super().__init__()
        mid_ch = out_ch // 2 if halve_channel else out_ch
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode=mode)
            self.conv = nn.Sequential(
                DoubleConv(in_ch + skip_ch, mid_ch, bn_first=bn_first),
                *[DoubleConv(mid_ch, mid_ch) for _ in range(n - 1)]
            )
        else:
            self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
            self.conv = nn.Sequential(
                DoubleConv(in_ch // 2 + skip_ch, mid_ch, bn_first=bn_first),
                *[DoubleConv(mid_ch, mid_ch) for _ in range(n - 1)]
            )
    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x): return self.conv(x)

class TrackNetV2(nn.Module):
    def __init__(self, n_channels=9, n_classes=3):
        super().__init__()
        self.inc   = DoubleConv(n_channels, 64, bn_first=False)
        self.down1 = Down(2, 64, 128, bn_first=False)
        self.down2 = Down(3, 128, 256, bn_first=False)
        self.down3 = Down(3, 256, 512, bn_first=False)
        self.up1   = Up(3, 512, 256, 256, bn_first=False)
        self.up2   = Up(2, 256, 128, 128, bn_first=False)
        self.up3   = Up(2, 128, 64, 64, bn_first=False)
        self.outc  = OutConv(64, n_classes)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x  = self.up1(x4, x3)
        x  = self.up2(x, x2)
        x  = self.up3(x, x1)
        return {0: self.outc(x)}

print('Modelo definido.')

In [ ]:
# Función de inferencia sobre vídeo
import cv2
import numpy as np
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
INP_H, INP_W = 288, 512
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
SCORE_THRESHOLD = 0.5  # Umbral para considerar pelota detectada

def preprocess_frame(frame):
    """BGR frame → tensor normalizado (3, H, W)"""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (INP_W, INP_H)).astype(np.float32) / 255.0
    normalized = (resized - MEAN) / STD
    return torch.from_numpy(normalized.transpose(2, 0, 1))  # (3, H, W)

def load_model(weights_path):
    model = TrackNetV2(n_channels=9, n_classes=3).to(DEVICE)
    checkpoint = torch.load(weights_path, map_location=DEVICE)
    state = checkpoint.get('model_state_dict', checkpoint)
    # Quitar prefijo 'module.' si viene de DataParallel
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    return model

def run_inference(model, video_path, max_frames=500, output_path=None, score_threshold=SCORE_THRESHOLD):
    cap = cv2.VideoCapture(video_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), max_frames or 999999)

    out = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_buffer = []  # Buffer de 3 tensores preprocesados
    raw_buffer   = []  # Buffer de 3 frames originales para anotar
    detected = 0
    processed = 0
    detections = []  # [(frame_idx, x, y, score)]

    pbar = tqdm(total=total, desc='Procesando')
    frame_idx = 0

    while cap.isOpened() and (max_frames == 0 or frame_idx < max_frames):
        ret, frame = cap.read()
        if not ret:
            break

        tensor = preprocess_frame(frame)
        frame_buffer.append(tensor)
        raw_buffer.append(frame)

        if len(frame_buffer) == 3:
            # Stack 3 frames → (1, 9, H, W)
            inp = torch.stack(frame_buffer, dim=0)       # (3, 3, H, W)
            inp = inp.view(1, 9, INP_H, INP_W).to(DEVICE)  # (1, 9, H, W)

            with torch.no_grad():
                output = model(inp)[0]                   # (1, 3, H, W)
                heatmap = torch.sigmoid(output[0, -1])   # Último frame → (H, W)

            hm_np = heatmap.cpu().numpy()
            score = hm_np.max()
            processed += 1

            ball_pos = None
            if score >= score_threshold:
                detected += 1
                # Convertir coords del heatmap al frame original
                hy, hx = np.unravel_index(hm_np.argmax(), hm_np.shape)
                bx = int(hx / INP_W * width)
                by = int(hy / INP_H * height)
                ball_pos = (bx, by, score)
                detections.append((frame_idx, bx, by, float(score)))

            # Anotar el frame del medio (frame_idx - 1)
            if out:
                annotated = raw_buffer[1].copy()
                if ball_pos:
                    bx, by, sc = ball_pos
                    cv2.circle(annotated, (bx, by), 12, (0, 255, 0), 3)
                    cv2.putText(annotated, f'{sc:.2f}', (bx+15, by),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                det_rate = detected / processed * 100
                cv2.putText(annotated, f'Det: {det_rate:.1f}% | Score: {score:.3f}',
                            (12, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (200, 200, 200), 2)
                out.write(annotated)

            # Ventana deslizante: quitar el más antiguo
            frame_buffer.pop(0)
            raw_buffer.pop(0)

        frame_idx += 1
        pbar.update(1)

    cap.release()
    if out:
        out.release()
    pbar.close()

    detection_rate = detected / processed * 100 if processed > 0 else 0
    return {
        'frames_procesados': processed,
        'frames_detectados': detected,
        'detection_rate': detection_rate,
        'detections': detections,
    }

print('Funciones de inferencia listas.')

In [ ]:
# --- COMPARATIVA DE MODELOS ---
# Ajusta MAX_FRAMES para test rápido (500 = ~16s de vídeo a 30fps)
MAX_FRAMES = 500

resultados = {}

modelos_a_probar = [
    ('TrackNetV2 Tennis',    'pretrained_weights/tracknetv2_tennis_best.pth.tar'),
    ('TrackNetV2 Badminton', 'pretrained_weights/tracknetv2_badminton_best.pth.tar'),
    ('WASB Tennis',          'pretrained_weights/wasb_tennis_best.pth.tar'),
    ('WASB Badminton',       'pretrained_weights/wasb_badminton_best.pth.tar'),
]

for nombre, weights_path in modelos_a_probar:
    print(f'\n=== {nombre} ===')
    try:
        model = load_model(weights_path)
        output_video = f'/tmp/test_{nombre.replace(" ", "_").lower()}.mp4'
        result = run_inference(model, VIDEO_PATH, max_frames=MAX_FRAMES, output_path=output_video)
        resultados[nombre] = result
        print(f'  Detection rate: {result["detection_rate"]:.1f}% ({result["frames_detectados"]}/{result["frames_procesados"]} frames)')
        del model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  ERROR: {e}')
        resultados[nombre] = {'error': str(e)}

In [ ]:
# Resumen comparativo
print('\n' + '='*50)
print('COMPARATIVA FINAL (referencia: YOLOv8 actual = ~20%)')
print('='*50)
for nombre, r in resultados.items():
    if 'error' in r:
        print(f'  {nombre:30s} → ERROR: {r["error"]}')
    else:
        rate = r['detection_rate']
        emoji = '✅' if rate > 60 else ('⚠️' if rate > 30 else '❌')
        print(f'  {nombre:30s} → {rate:5.1f}%  {emoji}')
print('='*50)

In [ ]:
# Descargar el mejor vídeo anotado para revisión visual
from google.colab import files

# Buscar el modelo con mejor detection rate
mejor = max(
    [(n, r) for n, r in resultados.items() if 'error' not in r],
    key=lambda x: x[1]['detection_rate'],
    default=(None, None)
)

if mejor[0]:
    mejor_nombre = mejor[0]
    mejor_path = f'/tmp/test_{mejor_nombre.replace(" ", "_").lower()}.mp4'
    print(f'Descargando vídeo del mejor modelo: {mejor_nombre} ({mejor[1]["detection_rate"]:.1f}%)')
    files.download(mejor_path)
else:
    print('No hay vídeos disponibles para descargar.')